# Figure 6: 3D Brain Connectome with LRG Clusters

This notebook generates the brain connectome visualization showing SEEG electrodes embedded in a 3D glass brain, with nodes colored by LRG cluster and edges showing functional connectivity.

**Outputs:**
- `fig6_brain_connectome_beta.html` - Interactive 3D brain (nilearn)
- `fig6_brain_connectome_beta_rsPre_multiview.png` - Static multi-view for publication

In [1]:
# Setup
from lrgsglib.config.funcs import move_to_rootf
move_to_rootf(pathname='lrgeegfc')

from lrg_eegfc.notebook import *
path_figs = setup_notebook('figures/presentation_figures')

Current working directory: /home/giulio/Documents/research/neural_networks/lrgeegfc
✓ Notebook setup complete
  Figure output: data/figures/presentation_figures


In [2]:
# Imports
import numpy as np
from pathlib import Path
from scipy.cluster.hierarchy import fcluster

# Spatial visualization (nilearn-based)
from lrg_eegfc.visuals.spatial import (
    load_spatial_metadata,
    prepare_spatial_coordinates,
    view_brain_connectome,
    plot_brain_connectome,
)

## Configuration

In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================
PATIENT = "Pat_02"
PHASES = ["rsPre", "rsPost"]
BAND = "beta"  # Frequency band
FC_METHOD = "msc"

# Visualization parameters
EDGE_THRESHOLD = 0.3  # Minimum edge weight to display
EDGE_POWER = 2.0  # Power-law exponent for edge width scaling (higher = more contrast)
NODE_SIZE = 8  # For Plotly
NODE_SIZE_MPL = 50  # For Matplotlib

# Paths
DATASET_ROOT = Path("data/stereoeeg_patients")
MSC_CACHE = Path("data/msc_cache")
LRG_CACHE = Path("data/lrg_cache")
OUTPUT_DIR = path_figs / PATIENT
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Patient: {PATIENT}")
print(f"Phases: {PHASES}")
print(f"Band: {BAND}")
print(f"Edge power-law exponent: {EDGE_POWER}")
print(f"Output: {OUTPUT_DIR}")

## Load spatial metadata

In [4]:
# Load electrode metadata with coordinates
metadata = load_spatial_metadata(PATIENT, DATASET_ROOT)

print(f"Loaded metadata for {len(metadata)} channels")
print(f"Columns: {list(metadata.columns)}")
print(f"\nFirst 5 channels:")
metadata.head()

Loaded metadata for 117 channels
Columns: ['label', 'x', 'y', 'z', 'Desikan-Killany']

First 5 channels:


,label,x,y,z,Desikan-Killany
0,"A 1,G2",-20528.0,-53975.0,-41970.0,"Cerebellum-Cortex,50.0,ctx-lh-lingual,36.0,Un..."
1,"A 2,G2",-24797.0,-55764.0,-42496.0,"ctx-lh-fusiform,80.0,Cerebellum-Cortex,12.0,W..."
2,"A 3,G2",-28290.0,-55616.0,-42656.0,"ctx-lh-fusiform,80.0,Wm,20.0,PTD, 0.61"
3,"A 4,G2",-31783.0,-55469.0,-42815.0,"ctx-lh-fusiform,80.0,Wm,11.0,Unk,7.0,Cerebell..."
4,"A 5,G2",-35276.0,-55321.0,-42974.0,"ctx-lh-fusiform,52.0,Wm,48.0,PTD, 0.04"


In [5]:
# Prepare coordinates (transform to MNI-like space for visualization)
coords_native = prepare_spatial_coordinates(metadata, scale="mm", center=True, to_mni=False)
coords_mni = prepare_spatial_coordinates(metadata, scale="mm", center=False, to_mni=True)

print(f"Native coordinates (centered): shape {coords_native.shape}")
print(f"  X range: [{coords_native[:,0].min():.1f}, {coords_native[:,0].max():.1f}] mm")
print(f"  Y range: [{coords_native[:,1].min():.1f}, {coords_native[:,1].max():.1f}] mm")
print(f"  Z range: [{coords_native[:,2].min():.1f}, {coords_native[:,2].max():.1f}] mm")

print(f"\nMNI-like coordinates: shape {coords_mni.shape}")
print(f"  X range: [{coords_mni[:,0].min():.1f}, {coords_mni[:,0].max():.1f}] mm")
print(f"  Y range: [{coords_mni[:,1].min():.1f}, {coords_mni[:,1].max():.1f}] mm")
print(f"  Z range: [{coords_mni[:,2].min():.1f}, {coords_mni[:,2].max():.1f}] mm")

Native coordinates (centered): shape (117, 3)
  X range: [-26.4, 32.9] mm
  Y range: [-32.4, 62.1] mm
  Z range: [-19.4, 24.1] mm

MNI-like coordinates: shape (117, 3)
  X range: [-63.0, -3.7] mm
  Y range: [-62.5, 32.0] mm
  Z range: [-26.7, 16.7] mm


## Load LRG results for cluster coloring

In [6]:
# Load LRG results and compute clusters with LOWER threshold for more communities
from matplotlib import cm

lrg = load_lrg_result(PATIENT, "rsPre", BAND, FC_METHOD, LRG_CACHE)

# Use lower threshold for finer partition (more clusters)
# Optimal threshold gives 3 clusters; threshold=0.05 gives 8 clusters
CLUSTER_THRESHOLD = 0.05  # Lower = more clusters

cluster_labels = fcluster(
    lrg.linkage_matrix,
    t=CLUSTER_THRESHOLD,
    criterion="distance"
)

n_clusters = len(np.unique(cluster_labels))
print(f"Using threshold={CLUSTER_THRESHOLD} -> {n_clusters} clusters")
print(f"(Optimal threshold={lrg.optimal_threshold:.4f} would give 3 clusters)")

# Create node colors from cluster labels
cmap = cm.get_cmap("tab10" if n_clusters <= 10 else "tab20")
unique_clusters = np.unique(cluster_labels)
color_map = {c: cmap(i / max(len(unique_clusters) - 1, 1)) for i, c in enumerate(unique_clusters)}
node_colors = [color_map[c] for c in cluster_labels]

print(f"\nCluster sizes:")
for c in unique_clusters:
    count = np.sum(cluster_labels == c)
    print(f"  Cluster {c}: {count} nodes")

Using threshold=0.05 -> 8 clusters
(Optimal threshold=0.3644 would give 3 clusters)

Cluster sizes:
  Cluster 1: 6 nodes
  Cluster 2: 8 nodes
  Cluster 3: 5 nodes
  Cluster 4: 3 nodes
  Cluster 5: 16 nodes
  Cluster 6: 65 nodes
  Cluster 7: 5 nodes
  Cluster 8: 9 nodes


/tmp/ipykernel_496327/3972542809.py:21: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  cmap = cm.get_cmap("tab10" if n_clusters <= 10 else "tab20")


## Glass Brain Connectome (nilearn)

In [ ]:
# Create main Figure 6: 3D Brain with embedded electrodes and connectivity
# Using nilearn's interactive glass brain view with power-law edge scaling

from nilearn.plotting import view_connectome
from lrg_eegfc.workflow.msc import load_msc_matrix

# Load FC matrix and apply power-law scaling
fc_matrix = load_msc_matrix(PATIENT, "rsPre", BAND, cache_root=MSC_CACHE)
fc_scaled = np.power(fc_matrix, EDGE_POWER)  # Power-law: emphasize strong connections

# Common edge colormap
EDGE_CMAP = "hot"

# Create interactive view with scaled matrix
view = view_connectome(
    adjacency_matrix=fc_scaled,
    node_coords=coords_mni,
    edge_threshold="70%",  # Show top 30% strongest edges
    edge_cmap=EDGE_CMAP,
    symmetric_cmap=False,  # MSC is always positive
    linewidth=6.0,  # Base linewidth (scaled by weight)
    node_size=10.0,
    node_color=node_colors,  # Use our custom cluster colors (8 clusters)
    colorbar=True,
    title=f"{PATIENT} {BAND.upper()} rsPre - Brain Connectome ({n_clusters} LRG Clusters)",
)

# Save as main Figure 6 HTML
html_path = OUTPUT_DIR / f"fig6_brain_connectome_{BAND}.html"
view.save_as_html(str(html_path))
print(f"Saved: {html_path}")
print(f"Edge weights scaled with power={EDGE_POWER} (emphasizes strong connections)")

# Display in notebook
view

In [ ]:
# Create static glass brain plot (multi-view) with power-law edge scaling
from nilearn.plotting import plot_connectome
import matplotlib.pyplot as plt

output_path = OUTPUT_DIR / f"fig6_brain_connectome_{BAND}_rsPre_multiview.png"

# Create figure
fig = plt.figure(figsize=(14, 4))

# Plot connectome with scaled matrix
plot_connectome(
    adjacency_matrix=fc_scaled,  # Use power-law scaled matrix
    node_coords=coords_mni,
    edge_threshold=EDGE_THRESHOLD,
    edge_cmap=EDGE_CMAP,  # Same colormap as HTML plot
    node_color=node_colors,  # Use our custom cluster colors (8 clusters)
    node_size=50.0,
    display_mode="lzry",  # Left, axial, right, coronal
    colorbar=True,
    title=f"{PATIENT} {BAND.upper()} rsPre ({n_clusters} clusters)",
    figure=fig,
)

fig.savefig(output_path, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {output_path}")